In [1]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


In [4]:
def deterministic_guardrail(text: str)->bool:
    """Return true if content is blocked."""
    banned_keywords=["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs=[
    "how do i hack into a databse?",
    "what is the capital of france?",
    "explain how maleware spreads"
]

print("==Deterministic Guardrail Demo==")
for inp in test_inputs:
    blocked=deterministic_guardrail(inp)
    status= "BLOCKED" if blocked else "ALLOWED"
    print(f"{status}: {inp}")

==Deterministic Guardrail Demo==
BLOCKED: how do i hack into a databse?
ALLOWED: what is the capital of france?
ALLOWED: explain how maleware spreads


In [3]:
def safety_guard(text: str)->bool:
    """Block text if it contains unsafe terms"""
    blocked_terms=["virus", "phishing", "ransomware", "trojan"]
    lower_text=text.lower()
    for term in blocked_terms:
        if term in lower_text:
            return True
    return False

print(safety_guard("This email contains a phishing attemt")) # True

True


In [5]:
from langchain_openai import ChatOpenAI

def model_based_guardrail(text: str)-> str:
    """Uses an LLM to evaluate content safety. Return SAFE or UNSAFE"""
    model=ChatOpenAI(model="gpt-4o-mini", temperature=0)
    prompt=f"""Is the following user input safe to process?
    Reply with only 'true' or 'false'
    Input: {text}"""
    result = model.invoke([{"role":"user","content":prompt}])
    return result.content.strip()

print("==Model Based Guardrails Demo==")
for inp in test_inputs:
    verdict=model_based_guardrail(inp)
    status="False" if "False" in verdict else "True"
    print(f"{status}: {inp}")

==Model Based Guardrails Demo==
True: how do i hack into a databse?
True: what is the capital of france?
True: explain how maleware spreads


In [6]:
from openai import OpenAI
client = OpenAI()
def model_guardrail_check(user_input: str) -> bool:
    prompt = f"""
    Check if the following input is SAFE or UNSAFE.
    Rules:
    - Unsafe: harmful, abusive, illegal, toxic content
    - Safe: normal, educational, harmless queries

    Input: "{user_input}"
    Answer only: SAFE or UNSAFE
    """
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    result = response.choices[0].message.content.strip()
    return result == "SAFE"
# Example usage
user_query = "How to hack a system?"
if model_guardrail_check(user_query):
    print("Allowed")
else:
    print("Blocked")

Blocked


In [8]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

@tool
def customer_lookup(query: str)-> str:
    """Look up customer lookup"""
    return f"Customer record found for query: {query}"

# Create agent for PII Middleware
agent=create_agent(
    model="gpt-4o-mini",
    tools=[customer_lookup],
    middleware=[
        # redact email in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block api keys
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

In [9]:
result=agent.invoke({
    "messages": [{
        "role":"user",
        "content":(
            "My email is john.doe@example.com and my card is "
            "5105-1051-0510-5100. Can you help me?"
        )
    }]
})
print("==Agent response==")
print(result["messages"][-1].content)


The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
==Agent response==
I found your customer record associated with your email. How can I assist you further?


In [10]:
try:
    result= agent.invoke({
        "messages": [{
            "role":"user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
except Exception as e:
    print(f"Blocked as expected: {e}")

Blocked as expected: Detected 1 instance(s) of api_key in text content


In [11]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query: str)-> str:
    """Search the web for information"""
    return f"search result for: {query}"

@tool
def send_email(to: str, subject: str, body: str)-> str:
    """Send an email to the recipient"""
    return f"Email send to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str)-> str:
    """Delete records from the database"""
    return f"Deleted records from {table} where {condition}"

# Create agent with HITL Middleware
agent=create_agent(
    model="gpt-4o-mini",
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "search_web":False,
                "send_email":True,
                "delete_records":True
            }
        ),
    ],
    checkpointer=InMemorySaver(),

)
